# Meet in the Middle

A divide-and-conquer trick for **exponential** search spaces that are too big for brute force but small enough to halve. Splitting n items into two halves turns **2ⁿ** work into **2 · 2^(n/2)** — the difference between hopeless (2⁴⁰ ≈ 10¹²) and instant (2²⁰ ≈ 10⁶). The classic use is **subset sum for n ≈ 40**.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **Split & combine**
3. **Combine smarter with `bisect`**
4. **When to use & cheat-sheet**

## 1. The signal — too big for 2ⁿ, fine for 2^(n/2)

Reach for meet-in-the-middle when:

- the natural solution is to **enumerate an exponential space** (all 2ⁿ subsets, all assignments), and
- **n is moderate** — too large for 2ⁿ (n > ~25) but small enough that **2^(n/2)** is fine (n up to ~40–45), and
- the space **splits**: the answer combines an independent choice over the first half with one over the second.

The move: enumerate each half separately (2^(n/2) each), then **combine** the two halves cheaply — with a hash set (for "exists / equals") or by sorting one side and binary-searching it (for "closest / count ≤").

## 2. Split & combine

The template: build all subset sums of the **left** half and all of the **right** half. For "does any subset sum to `target`?", store the left sums in a set, then for each right sum `s` check whether `target - s` is present. Two halves of 2^(n/2) work, instead of one pass of 2ⁿ:

In [1]:
def all_subset_sums(nums):
    sums = {0}
    for x in nums:
        sums |= {s + x for s in sums}        # each item doubles the set of reachable sums
    return sums

def subset_sum_exists(nums, target):
    mid = len(nums) // 2
    left = all_subset_sums(nums[:mid])       # 2^(n/2) sums
    for s in all_subset_sums(nums[mid:]):    # 2^(n/2) sums
        if (target - s) in left:             # O(1) lookup -> a matching pair exists
            return True
    return False

print("exists([3,34,4,12,5,2], 9):", subset_sum_exists([3, 34, 4, 12, 5, 2], 9))
print("exists([3,34,4,12,5,2], 1):", subset_sum_exists([3, 34, 4, 12, 5, 2], 1))


exists([3,34,4,12,5,2], 9): True
exists([3,34,4,12,5,2], 1): False


`all_subset_sums` doubles its set per item, so it's 2^m for m items — which is exactly why we only ever run it on a **half** of the input. The whole trick is paying 2 × 2^(n/2) instead of 2ⁿ.

## 3. Combine smarter with `bisect`

When the question is "**closest** sum to target" (or "how many ≤ target"), an exact hash lookup won't do — you need the *nearest* value. Sort one half's sums once, then for each sum in the other half **binary-search** for the best complement. That's the searching notebook's `bisect`, combining the halves in O(2^(n/2) · log 2^(n/2)):

In [2]:
import bisect

def closest_subset_sum(nums, target):
    mid = len(nums) // 2
    A = sorted(all_subset_sums(nums[:mid]))
    best, best_sum = float("inf"), None
    for b in all_subset_sums(nums[mid:]):
        i = bisect.bisect_left(A, target - b)        # where the ideal complement would sit
        for j in (i, i - 1):                         # check the neighbors on both sides
            if 0 <= j < len(A):
                total = A[j] + b
                if abs(total - target) < best:
                    best, best_sum = abs(total - target), total
    return best_sum

print("closest([7,11,19,23], 20):", closest_subset_sum([7, 11, 19, 23], 20))   # 19

# the whole point: this scales to n where 2^n enumeration is impossible
import random
random.seed(0)
big = [random.randint(1, 10**6) for _ in range(34)]   # 2^34 brute force: ~17 billion
target = sum(big) // 3
ans = closest_subset_sum(big, target)                  # 2^17 per half -> instant
print(f"n=34 -> closest to {target}: {ans} (off by {abs(ans - target)})")


closest([7,11,19,23], 20): 19
n=34 -> closest to 6436681: 6436681 (off by 0)


## 4. When to use & cheat-sheet

| You have… | Meet-in-the-middle… |
|---|---|
| subset sum / partition, n ≈ 30–45 | splits 2ⁿ into 2 · 2^(n/2) |
| "closest subset sum to target" | sort one half + `bisect` the other |
| "count subsets with sum ≤ target" | sort one half + `bisect` (count a prefix) |
| 4-sum / discrete log on big ranges | the same halve-and-combine idea |

**Python toolkit:** a `set` for exact-match combines; `sorted(...)` + `bisect` for nearest / count combines; the per-half enumeration is the bitmask/subset-sum doubling from the bit-manipulation and combinatorial-generation notebooks.

**The boundary:** below n ≈ 25, plain 2ⁿ enumeration is simpler and fast enough. Above n ≈ 45, even 2^(n/2) is too much — switch to **dynamic programming** (if the sums are bounded → the knapsack-style DP from the DP notebook) or an approximation. Meet-in-the-middle owns the awkward middle.

| Phase | Work |
|---|:---:|
| Enumerate each half | 2 × O(2^(n/2)) |
| Combine — exact match | O(2^(n/2)) with a set |
| Combine — closest / count | O(2^(n/2) · n) with sort + bisect |
| Brute force *(for contrast)* | O(2ⁿ) |